In [1]:
# %% Shared setup
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Tapiculin"
STUDENT_ID = "TUPM-26-1932"
id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)

WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

X = np.genfromtxt(DATA_DIR / "secom.data", dtype=float)          # measurement matrix
labels = np.loadtxt(DATA_DIR / "secom_labels.data", usecols=0, dtype=int)
assert len(labels) == len(X)
feature_names = [f"Feature_{i:03d}" for i in range(X.shape[1])]

missing = np.isnan(X)
observed_n = (~missing).sum(axis=0)
obs_min = np.where(missing, np.inf, X).min(axis=0)
obs_max = np.where(missing, -np.inf, X).max(axis=0)
all_missing = observed_n == 0
constant = (observed_n > 0) & (obs_min == obs_max)
usable = np.flatnonzero(~(all_missing | constant))               # at least two distinct observed values
print("Records:", X.shape[0], "| Features:", X.shape[1], "| Usable features:", len(usable))
print("Seed digit:", SEED_DIGIT, "| ID sum:", ID_SUM, "| Random seed:", RANDOM_SEED)


# %% EXERCISE 1 - Vectorized Measurement Quality Analysis (NumPy only)
rng = np.random.default_rng(RANDOM_SEED)
n_select = 5 + (SEED_DIGIT % 3)
sel = np.sort(rng.choice(usable, size=n_select, replace=False))
S = X[:, sel]
S_miss = np.isnan(S)

# 3. Feature-wise statistics (vectorized)
means = np.nanmean(S, axis=0)
sds = np.nanstd(S, axis=0, ddof=1)
miss_pct = 100 * S_miss.mean(axis=0)

# 4. Robust standardization with median and MAD (zero MAD handled safely)
medians = np.nanmedian(S, axis=0)
mad = np.nanmedian(np.abs(S - medians), axis=0)
safe_mad = np.where(mad > 0, mad, np.nan)
robust_z = 0.6745 * (S - medians) / safe_mad                 # NaN where MAD == 0 or value missing

# 5. Flag |robust z| > 3.5 (NaN comparisons are False)
flags = np.abs(robust_z) > 3.5
n_obs = (~S_miss).sum(axis=0)
flag_pct = 100 * flags.sum(axis=0) / np.maximum(n_obs, 1)

# 6. Rank by flagged percentage and export
summary = pd.DataFrame({
    "Feature": [feature_names[i] for i in sel], "Mean": means, "SampleSD": sds,
    "MissingPct": miss_pct, "Median": medians, "MAD": mad,
    "FlaggedCount": flags.sum(axis=0), "FlaggedPct": flag_pct,
}).sort_values("FlaggedPct", ascending=False).reset_index(drop=True)
summary.insert(0, "Rank", summary.index + 1)
summary.round(4).to_csv(OUTPUT_DIR / "measurement_quality.csv", index=False)

# Verification of vectorized results against a plain loop on the first selected feature
col = S[:, 0][~np.isnan(S[:, 0])]
loop_mean = sum(col) / len(col)
loop_sd = (sum((v - loop_mean) ** 2 for v in col) / (len(col) - 1)) ** 0.5
verified = bool(np.isclose(loop_mean, means[0]) and np.isclose(loop_sd, sds[0]))

top = summary.iloc[0]
print(summary.round(4).to_string(index=False))
print("\n=== Assessment Data ===")
print("Selected Features:", [feature_names[i] for i in sel])
print("Highest Missing Percentage:", round(float(miss_pct.max()), 2), "%")
print("Feature with Highest Flagged Percentage:", top["Feature"], f"({top['FlaggedPct']:.2f}%)")
print("Total Flagged Observations:", int(flags.sum()))
print("Vectorized Results Verified:", verified)
print("Required Output: measurement_quality.csv ->", (OUTPUT_DIR / "measurement_quality.csv").is_file())

Records: 1567 | Features: 590 | Usable features: 474
Seed digit: 2 | ID sum: 23 | Random seed: 261932
 Rank     Feature      Mean  SampleSD  MissingPct    Median     MAD  FlaggedCount  FlaggedPct
    1 Feature_435    9.3707   34.3698      0.1276    4.5511  1.1519            45      2.8754
    2 Feature_108   -0.0108    0.0868      0.3829   -0.0112  0.0516            18      1.1531
    3 Feature_000 3014.4529   73.6218      0.3829 3011.4900 45.2300            12      0.7687
    4 Feature_472  137.8884   47.6980      0.4467  138.2552 31.3425             6      0.3846
    5 Feature_271   46.0566   17.8664      0.3829   45.6765 15.0279             1      0.0641
    6 Feature_055 2856.1721   25.7493      0.2553 2854.0000 19.0000             0      0.0000
    7 Feature_511  275.9795  329.6647      0.1276    0.0000  0.0000             0      0.0000

=== Assessment Data ===
Selected Features: ['Feature_000', 'Feature_055', 'Feature_108', 'Feature_271', 'Feature_435', 'Feature_472', 'Feature_51